# Native NVFP4 and MXFP8 on NVIDIA Blackwell (CUDA) - the hardware half of end-to-end 4-bit training
### Pack the operands, swizzle the scales, call the block-scaled tensor-core GEMMs, and check them against the emulated reference

Companion to `01_end_to_end_4bit_training_on_macbook.ipynb`, which builds the numerics of the *End-to-End 4-Bit Training* recipe ([Thinking Machines Lab](https://thinkingmachines.ai/blog/end-to-end-4-bit-training/)) in emulation. This notebook runs the same recipe's matrix multiplies on **real FP4 / FP8 tensor cores**.

> **Status of this notebook - read first.**
> It was written and executed on a MacBook with **no NVIDIA GPU**. Every cell that needs a Blackwell GPU printed `SKIP` there; **those cells have never run on Blackwell hardware**. The CPU cells (packing, scale layout, containers, API plumbing) did run and pass. The GPU cells are built so that a mistake cannot pass silently: each one compares the native result to the emulated reference and reports `PASS` or `FAIL` with the measured difference, and the final cell lists what ran. **If you run it on a Blackwell GPU, the `FAIL`/`PASS` lines are the real test of this notebook.**

**Hardware that has the FP4 tensor cores:** data-centre Blackwell (B200, GB200, GB300 - compute capability 10.x) and consumer / workstation Blackwell (RTX 50-series, RTX PRO Blackwell - compute capability 12.x). **Software:** a PyTorch build with CUDA 12.8 or newer.

| step | where it runs | what it establishes |
|---|---|---|
| 0 | anywhere | what hardware/software you have, and whether the GPU cells will run |
| 1 | anywhere | PyTorch's FP4 / FP8 dtypes exist |
| 2 | anywhere (CPU) | FP4 packing (two values per byte) and the scale layout, checked against PyTorch's own reference helpers |
| 3 | anywhere (CPU) | the operand containers carry exactly the emulated tensors, at 4.5 / 8.25 bits per value |
| 4 | anywhere | the `scaled_mm` call signature is accepted by PyTorch (a weak, shape-only check); the whole GPU code path run on the CPU with a stand-in kernel |
| 5 | **Blackwell** | native NVFP4 and MXFP8 GEMM == emulated GEMM |
| 6 | **Blackwell** | the whole recipe layer (NVFP4 forward, MXFP8 backward) native == emulated |
| 7 | **Blackwell** | throughput: bf16 vs MXFP8 vs NVFP4 |
| 8 | **Blackwell** + libraries | the vendor implementations (Transformer Engine, torchao), if installed |
| 9 | **Blackwell** | a small network trained with native GEMMs vs the emulation |

In [ ]:
import math, sys, time, ast, traceback
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import fp4kit as k

torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

# ---- hardware probe -------------------------------------------------------------------------------------------------
HAS_CUDA = torch.cuda.is_available()
CC = torch.cuda.get_device_capability(0) if HAS_CUDA else None
CUDA_VER = tuple(int(v) for v in torch.version.cuda.split(".")[:2]) if torch.version.cuda else None
BLACKWELL = bool(HAS_CUDA and CC[0] in (10, 12) and CUDA_VER is not None and CUDA_VER >= (12, 8))
print("torch", torch.__version__, "| torch built for CUDA:", torch.version.cuda, "| HIP:", torch.version.hip)
if HAS_CUDA:
    p = torch.cuda.get_device_properties(0)
    kind = {10: "data-centre Blackwell (sm_10x: B200/GB200/GB300)", 12: "consumer/workstation Blackwell (sm_12x: RTX 50 / RTX PRO)"}.get(CC[0], "NOT Blackwell")
    print(f"GPU: {p.name} | compute capability {CC[0]}.{CC[1]} -> {kind} | {p.total_memory / 2**30:.0f} GiB")
else:
    print("GPU: none visible (this is the Mac / CPU-only case) - GPU cells will be skipped")
print("-> Blackwell GPU cells will run:", BLACKWELL)
if HAS_CUDA and not BLACKWELL:
    print("   reason:", "compute capability is not 10.x/12.x" if CC[0] not in (10, 12) else "torch CUDA build older than 12.8")

# ---- recording --------------------------------------------------------------------------------------------------------
CHECKS = []                      # (name, status, detail)   status in PASS / FAIL / SKIP
def check(name, ok, detail=""):
    CHECKS.append((name, "PASS" if ok else "FAIL", detail)); print(("PASS" if ok else "FAIL"), "-", name, ("| " + detail) if detail else "")
def gpu_check(name, fn):
    """Run fn() -> (ok, detail) only on Blackwell. Exceptions are recorded as FAIL with the message, never swallowed silently."""
    if not BLACKWELL:
        CHECKS.append((name, "SKIP", "needs a Blackwell GPU")); print("SKIP -", name, "| needs a Blackwell GPU"); return
    try:
        ok, detail = fn()
    except Exception as e:
        CHECKS.append((name, "FAIL", f"{type(e).__name__}: {str(e)[:400]}")); print("FAIL -", name, "|", f"{type(e).__name__}: {str(e)[:400]}"); return
    CHECKS.append((name, "PASS" if ok else "FAIL", detail)); print("PASS" if ok else "FAIL", "-", name, "|", detail)

## Step 1 - PyTorch's low-precision dtypes

Block-scaled GEMMs take **packed FP4** (two E2M1 values per byte), **FP8 E4M3** elements, and scale tensors in **E4M3** (NVFP4) or **E8M0** (MXFP8). PyTorch exposes them as `torch.float4_e2m1fn_x2`, `torch.float8_e4m3fn` and `torch.float8_e8m0fnu`.

In [ ]:
for name in ("float4_e2m1fn_x2", "float8_e4m3fn", "float8_e8m0fnu"):
    dt = getattr(torch, name, None)
    print(f"torch.{name:18s} exists: {dt is not None}   bytes/element: {torch.empty(1, dtype=dt).element_size() if dt is not None else '-'}")
check("torch has float4_e2m1fn_x2, float8_e4m3fn, float8_e8m0fnu", all(hasattr(torch, n) for n in ("float4_e2m1fn_x2", "float8_e4m3fn", "float8_e8m0fnu")))
check("float4_e2m1fn_x2 occupies 1 byte (it holds TWO fp4 values)", torch.empty(4, dtype=torch.float4_e2m1fn_x2).element_size() == 1)
check("torch.nn.functional.scaled_mm (recipe + swizzle API) is available", hasattr(F, "scaled_mm"), "else the notebook falls back to torch._scaled_mm" if not hasattr(F, "scaled_mm") else "")

## Step 2 - Packing FP4 and swizzling the scales

Two layout rules decide whether a native GEMM returns the right numbers. Both are easy to get wrong, so we check ours against **PyTorch's own reference helpers** (`torch/testing/_internal/common_quantized.py`; we read the functions from that file's source rather than importing it, because importing it pulls in a test-only dependency).

**Rule 1 - packing.** An FP4 value is a 4-bit code: 1 sign bit + a 3-bit index into `{0, .5, 1, 1.5, 2, 3, 4, 6}`. Two codes share a byte: **the even element in the low nibble, the odd element in the high nibble**.

**Rule 2 - scale layout.** The scale matrix (`rows × K/block`) is not passed row-major. cuBLAS wants it in *128 × 4 tiles stored as 32 × 16* ("swizzled", `SWIZZLE_32_4_4`). Element `(r, c)` of a tile lands at flat offset `(r mod 32)·16 + (r div 32)·4 + c`.

In [ ]:
def torch_reference():
    """PyTorch's reference quantization helpers, extracted from the source of torch/testing/_internal/common_quantized.py."""
    path = Path(torch.__file__).parent / "testing" / "_internal" / "common_quantized.py"
    text = path.read_text(); tree = ast.parse(text)
    want = {"_n_ones", "_f32_to_floatx_unpacked", "_floatx_unpacked_to_f32", "ceil_div", "to_blocked", "down_size", "pack_uint4", "to_mxfp", "_bfloat16_to_float4_e2m1fn_x2"}
    parts = [ast.get_source_segment(text, n) for n in tree.body if isinstance(n, ast.FunctionDef) and n.name in want]
    parts += ["EBITS_F32, MBITS_F32 = 8, 23", "F32_EXP_BIAS = _n_ones(EBITS_F32 - 1)", "FP4_EBITS, FP4_MBITS = 2, 1"]
    ns = {"torch": torch, "Tensor": torch.Tensor}; exec("\n".join(parts), ns); return ns
try:
    REF = torch_reference(); print("PyTorch reference helpers loaded:", sorted(n for n in REF if n.startswith(("_f", "pack", "to_", "ceil"))))
except Exception as e:
    REF = None; print("reference helpers unavailable in this torch build:", type(e).__name__, e)

torch.manual_seed(0)
q = k.round_fp4(torch.randn(8, 64) * 3)                       # values on the FP4 grid
packed = k.pack_fp4(q)
print("fp4 values", q.shape, "-> packed bytes", packed.shape, packed.dtype, "| first row, first 4 values:", q[0, :4].tolist(), "-> bytes", [hex(int(b)) for b in packed[0, :2]])
check("pack/unpack round trip is exact", torch.equal(k.unpack_fp4(packed), q))
check("even element sits in the LOW nibble (bytes 0x?0 = high nibble only for index 0 == 0.0)", int(k.pack_fp4(torch.tensor([[1.0, 0.0]]))[0, 0]) == 0x02 and int(k.pack_fp4(torch.tensor([[0.0, 1.0]]))[0, 0]) == 0x20)
if REF is not None:
    codes = REF["_f32_to_floatx_unpacked"](q, 2, 1)
    check("fp4 codes equal PyTorch's _f32_to_floatx_unpacked", torch.equal(k.fp4_codes(q), codes))
    check("packed bytes equal PyTorch's pack_uint4 (low nibble = even element)", torch.equal(packed, REF["pack_uint4"](codes)))
    for shape in [(128, 4), (200, 10), (512, 64)]:
        s = torch.randn(*shape)
        check(f"scale swizzle equals PyTorch's to_blocked, scale matrix {shape}", torch.equal(k.to_blocked(s), REF["to_blocked"](s)))

In [ ]:
# the swizzle as a picture: where each of the 128 x 4 scale positions of a tile lands (colour = destination offset)
idx = torch.arange(128 * 4, dtype=torch.float32).reshape(128, 4)
blocked = k.to_blocked(idx).reshape(32, 16)
dest = torch.empty(128, 4)
for r in range(128):
    for c in range(4): dest[r, c] = (r % 32) * 16 + (r // 32) * 4 + c
check("formula: offset(r, c) = (r mod 32)*16 + (r div 32)*4 + c reproduces to_blocked", torch.equal(k.to_blocked(idx), idx.flatten()[torch.argsort(dest.flatten())]))
fig, axs = plt.subplots(1, 2, figsize=(9, 5.2), gridspec_kw={"width_ratios": [4, 16]})
axs[0].imshow(dest, aspect="auto", cmap="viridis"); axs[0].set_title("scale tile as stored by you\n(128 rows x 4 cols)\ncolour = destination offset", fontsize=9)
axs[0].set_xlabel("col"); axs[0].set_ylabel("row")
axs[1].imshow(idx.flatten()[torch.argsort(dest.flatten())].reshape(32, 16).div(4).floor(), aspect="auto", cmap="tab20"); axs[1].set_title("swizzled 32 x 16 layout\ncolour = source row", fontsize=9)
axs[1].set_xlabel("4 groups of (4 cols) = rows r, r+32, r+64, r+96"); plt.tight_layout(); plt.show()

## Step 3 - Operand containers: what the GEMM actually receives

`nvfp4_parts(x)` turns a 2-D tensor into the exact things an NVFP4 GEMM consumes: **packed FP4 bytes, E4M3 block scales, an FP32 outer scale**. `mxfp8_parts(x)` does the same for MXFP8 (E4M3 elements, E8M0 scales). Two properties to establish on the CPU, because they hold regardless of the GPU:

1. **Nothing is lost**: de-quantizing the parts reproduces the emulated tensor from notebook 01 exactly.
2. **The cost is real**: counting the bytes of the actual tensors gives 4.5 and 8.25 bits per value, as in the post's table.

In [ ]:
torch.manual_seed(1)
M, K = 256, 512
x = torch.randn(M, K) * (1 + 9 * (torch.rand(1, K) < 0.03))
for label, kw in [("tensor-scale weights", dict()), ("per-token activations + 4/6", dict(four_over_six=True, outer="row"))]:
    parts = k.nvfp4_parts(x, **kw)
    emu = k.quantize_nvfp4(x, **kw).dequantize()
    check(f"NVFP4 parts ({label}) dequantize == emulated tensor", torch.allclose(parts.dequantize(), emu, atol=1e-6, rtol=1e-6))
bits = 8 * (parts.packed.numel() * 1 + parts.delta.numel() * 1) / x.numel()          # 1 byte per packed pair, 1 byte (E4M3) per block scale
check("NVFP4 operand = 4.5 bits per value (packed fp4 + one E4M3 scale per 16)", bits == 4.5, f"{bits}")
mp = k.mxfp8_parts(x)
check("MXFP8 parts dequantize == emulated MXFP8", torch.equal(mp.dequantize(), k.quantize_mx(x, "mxfp8")))
bits8 = 8 * (mp.q.numel() * 1 + mp.scale.numel() * 1) / x.numel()
check("MXFP8 operand = 8.25 bits per value (fp8 + one E8M0 scale per 32)", bits8 == 8.25, f"{bits8}")
q8, sc8 = mp.hw_operands()
check("hardware casts are exact: float8_e4m3fn elements == emulated values", torch.equal(q8.float(), mp.q))
pk, sc4 = k.nvfp4_parts(x).hw_operands()
check("hardware dtypes: float4_e2m1fn_x2 elements, float8_e4m3fn scales", pk.dtype == torch.float4_e2m1fn_x2 and sc4.dtype == torch.float8_e4m3fn, f"packed {tuple(pk.shape)}, scales {sc4.numel()}")
print("operand shapes for a (256 x 512) tensor:  fp4x2", tuple(pk.shape), "| nvfp4 scales", sc4.numel(), "(padded to 128 rows x 4 col tiles) | fp8", tuple(q8.shape), "| e8m0 scales", sc8.numel())

## Step 4 - The GEMM call, and what can be checked without a GPU

The native matmul is one PyTorch call (`kit._scaled_mm`, shown below). `scaled_mm` takes the operands, the scale tensors, a **recipe** per operand (`BlockWise1x16` for NVFP4, `BlockWise1x32` for MXFP8) and the swizzle enum; the second operand is passed transposed (`b.t()`), i.e. column-major. The FP32 outer scales are applied to the output afterwards - exact, because they factor out of the sum.

In [ ]:
import inspect
print(inspect.getsource(k._scaled_mm))
print(inspect.getsource(k.gemm_nvfp4))

In [ ]:
# Argument-plumbing dry run on the 'meta' device (no memory, no GPU). IMPORTANT: this only shows that PyTorch accepts the call shape
# and returns the right output shape/dtype. The meta kernel does NOT validate scale sizes or dtypes (we verify that below), so this is
# a weak check; the real test is Step 5 on hardware.
S, W = F.ScalingType, F.SwizzleType
Mm, Nm, Km = 128, 128, 256
a = torch.empty(Mm, Km // 2, dtype=torch.float4_e2m1fn_x2, device="meta"); b = torch.empty(Nm, Km // 2, dtype=torch.float4_e2m1fn_x2, device="meta")
sa = torch.empty(k.to_blocked(torch.empty(Mm, Km // 16)).numel(), dtype=torch.float8_e4m3fn, device="meta")
out = F.scaled_mm(a, b.t(), sa, S.BlockWise1x16, sa.clone(), S.BlockWise1x16, swizzle_a=W.SWIZZLE_32_4_4, swizzle_b=W.SWIZZLE_32_4_4, output_dtype=torch.bfloat16)
check("meta dry run: NVFP4 scaled_mm accepted, output (128,128) bf16", out.shape == (Mm, Nm) and out.dtype == torch.bfloat16)
qa = torch.empty(Mm, Km, dtype=torch.float8_e4m3fn, device="meta"); sc = torch.empty(k.to_blocked(torch.empty(Mm, Km // 32)).numel(), dtype=torch.float8_e8m0fnu, device="meta")
out = F.scaled_mm(qa, qa.t(), sc, S.BlockWise1x32, sc.clone(), S.BlockWise1x32, swizzle_a=W.SWIZZLE_32_4_4, swizzle_b=W.SWIZZLE_32_4_4, output_dtype=torch.bfloat16)
check("meta dry run: MXFP8 scaled_mm accepted, output (128,128) bf16", out.shape == (Mm, Mm) and out.dtype == torch.bfloat16)
bad = torch.empty(7, dtype=torch.float32, device="meta")
try:
    F.scaled_mm(a, b.t(), bad, S.BlockWise1x16, bad.clone(), S.BlockWise1x16, swizzle_a=W.SWIZZLE_32_4_4, swizzle_b=W.SWIZZLE_32_4_4, output_dtype=torch.bfloat16); lenient = True
except Exception:
    lenient = False
print("meta kernel accepts a 7-element float32 'scale' tensor:", lenient, "-> it does not validate scale size/dtype; only hardware can")

### Step 4b - Run the GPU code path on the CPU with a stand-in kernel

The weakness of the meta check is that no numbers flow. So here the *same* `scaled_mm` code path - hardware dtypes, swizzled scales, outer scales, transposed second operand, the backward-pass re-quantization - is run with `_scaled_mm` replaced by `cpu_standin_scaled_mm`, which decodes exactly those packed bytes and swizzled scales and multiplies in float32. This verifies **our** formatting and bookkeeping end to end (and catches e.g. a swapped nibble order, shown below); it cannot verify cuBLAS itself.

In [ ]:
import contextlib
@contextlib.contextmanager
def standin_kernel():
    orig = k._scaled_mm; k._scaled_mm = k.cpu_standin_scaled_mm
    try: yield
    finally: k._scaled_mm = orig

torch.manual_seed(0)
x = torch.randn(256, 512) * (1 + 9 * (torch.rand(1, 512) < 0.03)); w = torch.randn(128, 512) / 22
a, b = k.nvfp4_parts(x, four_over_six=True, outer="row"), k.nvfp4_parts(w)
rel_ = lambda a_, b_: ((a_ - b_).norm() / b_.norm()).item()
with standin_kernel():
    r4 = rel_(k.gemm_nvfp4(a, b, "scaled_mm"), k.gemm_nvfp4(a, b, "emulated"))
    a8, b8 = k.mxfp8_parts(x), k.mxfp8_parts(w)
    r8 = rel_(k.gemm_mxfp8(a8, b8, "scaled_mm"), k.gemm_mxfp8(a8, b8, "emulated"))
check("stand-in kernel: NVFP4 scaled_mm path == emulated (bf16 output rounding only)", r4 < 1e-2, f"{r4:.2e}")
check("stand-in kernel: MXFP8 scaled_mm path == emulated", r8 < 1e-2, f"{r8:.2e}")

g = torch.randn(256, 128) * 0.01; outs = {}
for backend in ("emulated", "scaled_mm"):
    with standin_kernel():
        torch.manual_seed(5); xx = torch.randn(256, 256).requires_grad_(); ww = (torch.randn(128, 256) / 16).requires_grad_()
        torch.manual_seed(77); y = k.fp4_linear_packed(xx, ww, k.RecipeConfig(), backend); y.backward(g)
    outs[backend] = (y.detach(), xx.grad, ww.grad)
rr = [rel_(p_, q_) for p_, q_ in zip(outs["scaled_mm"], outs["emulated"])]
check("stand-in kernel: full recipe layer (fwd + both bwd GEMMs) scaled_mm path == emulated", max(rr) < 3e-2, f"y {rr[0]:.1e}, dx {rr[1]:.1e}, dW {rr[2]:.1e}")

swap = lambda p_: ((p_ & 0xF) << 4) | (p_ >> 4)
a_bad = k.NVFP4Parts(swap(a.packed), a.delta, a.alpha, a.shape)
with standin_kernel(): bad = rel_(k.gemm_nvfp4(a_bad, b, "scaled_mm"), k.gemm_nvfp4(a, b, "emulated"))
check("guard for the guard: a swapped nibble order is detected (error >> 1e-2)", bad > 0.3, f"{bad:.2f}")

## Step 5 - Native GEMM versus emulated GEMM *(Blackwell only)*

Same quantized operands, two ways to multiply them: dequantize and use a float32 matmul (the emulation), or hand the packed operands to the tensor cores. They should agree up to the **bf16 rounding of the GEMM output** (relative error ≈ 2·10⁻³) and accumulation order. A wrong nibble order or a wrong scale layout gives an error of order 1 - so a tolerance of 2 % cleanly separates "correct" from "wrong".

If the NVFP4 check fails, the diagnostic cell right after it tries the two most likely layout mistakes (swapped nibbles, un-swizzled scales) and reports which one matches.

In [ ]:
def rel(a, b): return ((a - b).norm() / b.norm()).item()
def make_operands(M, N, K, dev="cuda", seed=0):
    torch.manual_seed(seed)
    x = torch.randn(M, K, device=dev) * (1 + 9 * (torch.rand(1, K, device=dev) < 0.03)); w = torch.randn(N, K, device=dev) / math.sqrt(K)
    return x, w

def t_nvfp4():
    x, w = make_operands(512, 512, 1024)
    a, b = k.nvfp4_parts(x, four_over_six=True, outer="row"), k.nvfp4_parts(w, outer="tensor")
    ref, nat = k.gemm_nvfp4(a, b, "emulated"), k.gemm_nvfp4(a, b, "scaled_mm")
    r = rel(nat, ref); return r < 2e-2, f"relative difference native vs emulated = {r:.2e} (bf16 output rounding is ~2e-3)"
def t_mxfp8():
    x, w = make_operands(512, 512, 1024)
    a, b = k.mxfp8_parts(x), k.mxfp8_parts(w)
    ref, nat = k.gemm_mxfp8(a, b, "emulated"), k.gemm_mxfp8(a, b, "scaled_mm")
    r = rel(nat, ref); return r < 2e-2, f"relative difference native vs emulated = {r:.2e}"
gpu_check("native NVFP4 GEMM (scaled_mm, BlockWise1x16) == emulated", t_nvfp4)
gpu_check("native MXFP8 GEMM (scaled_mm, BlockWise1x32) == emulated", t_mxfp8)

In [ ]:
# Diagnostic - only does something if the NVFP4 check above failed on a Blackwell GPU.
failed = [c for c in CHECKS if c[0].startswith("native NVFP4 GEMM") and c[1] == "FAIL"]
if BLACKWELL and failed:
    x, w = make_operands(512, 512, 1024)
    a, b = k.nvfp4_parts(x, four_over_six=True, outer="row"), k.nvfp4_parts(w, outer="tensor"); ref = k.gemm_nvfp4(a, b, "emulated")
    swap = lambda p: ((p & 0xF) << 4) | (p >> 4)
    variants = {"as implemented": (a.packed, b.packed, True), "nibbles swapped": (swap(a.packed), swap(b.packed), True), "scales NOT swizzled": (a.packed, b.packed, False)}
    for name, (pa, pb, swz) in variants.items():
        try:
            sa_ = (k.to_blocked if swz else (lambda t: t.flatten()))(a.delta.to(torch.float8_e4m3fn)); sb_ = (k.to_blocked if swz else (lambda t: t.flatten()))(b.delta.to(torch.float8_e4m3fn))
            y = k._scaled_mm(pa.contiguous().view(torch.float4_e2m1fn_x2), sa_, "1x16", pb.contiguous().view(torch.float4_e2m1fn_x2), sb_, "1x16").float()
            oa = a.alpha.reshape(-1, 1); ob = b.alpha.reshape(1, 1)
            print(f"{name:22s} relative difference vs emulated: {rel(y * oa * ob, ref):.3e}")
        except Exception as e:
            print(f"{name:22s} raised {type(e).__name__}: {str(e)[:160]}")
else:
    print("diagnostic not needed (NVFP4 check did not fail or no Blackwell GPU)")

## Step 6 - The whole recipe layer, native versus emulated *(Blackwell only)*

`fp4_linear_packed(x, w, cfg, backend)` is the recipe from the post on packed operands: **forward** NVFP4 GEMM (4/6 on activations with per-token outer scales, per-weight-matrix scale, RTN); **backward** two MXFP8 GEMMs (`dx = g·w_q`, `dW = gᵀ·x_q`) after de-quantizing the saved forward tensors, transposing, and re-quantizing with stochastic rounding.

Run with `backend="emulated"` it reproduces `fp4_linear` exactly (a CPU unit test verifies this). Run with `backend="scaled_mm"` the three matmuls use the tensor cores. We seed the random generator identically so both backends draw **the same stochastic-rounding numbers**; then forward output, `dx` and `dW` must agree up to bf16 output rounding.

In [ ]:
def t_layer():
    T, K, N = 512, 512, 512
    torch.manual_seed(11); g = torch.randn(T, N, device="cuda") * 0.01
    outs = {}
    for backend in ("emulated", "scaled_mm"):
        torch.manual_seed(5)
        x = torch.randn(T, K, device="cuda").requires_grad_(); w = (torch.randn(N, K, device="cuda") / math.sqrt(K)).requires_grad_()
        torch.manual_seed(77)                                              # identical stochastic-rounding draws for both backends
        y = k.fp4_linear_packed(x, w, k.RecipeConfig(), backend); y.backward(g)
        outs[backend] = (y.detach().float(), x.grad.float(), w.grad.float())
    r = [rel(a, b) for a, b in zip(outs["scaled_mm"], outs["emulated"])]
    return max(r) < 3e-2, f"relative difference native vs emulated: y {r[0]:.2e}, dx {r[1]:.2e}, dW {r[2]:.2e}"
gpu_check("recipe layer: native forward + backward == emulated (same SR draws)", t_layer)

## Step 7 - Throughput *(Blackwell only)*

Only the GEMM is timed (operands are quantized beforehand): bf16 `x @ wᵀ` versus the two block-scaled GEMMs. The post quotes, for GB300, peak dense throughput of **2.5 PFLOPS (BF16), 5 (MXFP8) and 15 (NVFP4)**, i.e. peak ratios of 1 : 2 : 6. Measured ratios will be lower (memory-bound shapes, launch overhead) and **will differ on consumer cards**; what you see is whatever your GPU does - nothing in this notebook asserts a speed-up.

In [ ]:
def bench(fn, iters=30, warm=8):
    for _ in range(warm): fn()
    torch.cuda.synchronize(); s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(iters): fn()
    e.record(); torch.cuda.synchronize(); return s.elapsed_time(e) / iters / 1e3

def t_bench():
    rows = []
    for M, N, K in [(4096, 4096, 4096), (8192, 8192, 8192)]:
        torch.manual_seed(0); x = torch.randn(M, K, device="cuda"); w = torch.randn(N, K, device="cuda")
        xb, wb = x.bfloat16(), w.bfloat16()
        a8, b8 = k.mxfp8_parts(x), k.mxfp8_parts(w); qa, sa = a8.hw_operands(); qb, sb = b8.hw_operands()
        a4, b4 = k.nvfp4_parts(x), k.nvfp4_parts(w); pa, sa4 = a4.hw_operands(); pb, sb4 = b4.hw_operands()
        t_bf = bench(lambda: xb @ wb.t()); t_8 = bench(lambda: k._scaled_mm(qa, sa, "1x32", qb, sb, "1x32")); t_4 = bench(lambda: k._scaled_mm(pa, sa4, "1x16", pb, sb4, "1x16"))
        fl = 2 * M * N * K
        rows.append((f"{M}x{N}x{K}", fl / t_bf / 1e12, fl / t_8 / 1e12, fl / t_4 / 1e12))
    print(f"{'shape':16s}{'bf16 TFLOPS':>14s}{'MXFP8 TFLOPS':>15s}{'NVFP4 TFLOPS':>15s}{'MXFP8/bf16':>12s}{'NVFP4/bf16':>12s}")
    for n, b, m8, m4 in rows: print(f"{n:16s}{b:14.0f}{m8:15.0f}{m4:15.0f}{m8 / b:12.2f}{m4 / b:12.2f}")
    return all(v > 0 for r in rows for v in r[1:]), "timings printed above (GEMM only, operands pre-quantized)"
gpu_check("throughput benchmark ran (bf16 vs MXFP8 vs NVFP4 GEMM)", t_bench)

## Step 8 - Vendor implementations *(Blackwell + the library installed)*

You rarely hand-roll this for production. Two maintained implementations, shown here **as a smoke test** (each is guarded: skipped if the package is not installed). Names and arguments are from the libraries' documentation; this notebook was not able to run them.

- **NVIDIA Transformer Engine** - `NVFP4BlockScaling` recipe: 2D (16×16) weight scaling, random Hadamard transform on the weight-gradient inputs and stochastic rounding of gradients, all on by default per its docs. Requires SM 10.0+.
- **torchao** - `NVFP4DynamicActivationNVFP4WeightConfig` (inference / PTQ; prototype API whose names have changed between versions).

In [ ]:
def t_te():
    import transformer_engine.pytorch as te
    from transformer_engine.common.recipe import NVFP4BlockScaling
    recipe = NVFP4BlockScaling()
    layer = te.Linear(1024, 1024, params_dtype=torch.bfloat16).cuda()
    x = torch.randn(512, 1024, device="cuda", dtype=torch.bfloat16, requires_grad=True)
    ref = F.linear(x, layer.weight)
    ctx = te.autocast(enabled=True, recipe=recipe) if hasattr(te, "autocast") else te.fp8_autocast(enabled=True, fp8_recipe=recipe)
    with ctx: y = layer(x)
    y.float().pow(2).mean().backward()
    r = rel(y.float(), ref.float()); fin = bool(torch.isfinite(x.grad).all())
    return fin and r < 0.3, f"output relative difference vs bf16 {r:.3f} (4-bit noise expected), grads finite: {fin}"
try:
    import transformer_engine  # noqa
    have_te = True
except Exception:
    have_te = False
if have_te: gpu_check("Transformer Engine NVFP4BlockScaling forward+backward runs and is close to bf16", t_te)
else: CHECKS.append(("Transformer Engine NVFP4BlockScaling", "SKIP", "transformer_engine not installed")); print("SKIP - Transformer Engine not installed")

def t_ao():
    from torchao.quantization import quantize_
    from torchao.prototype.mx_formats import NVFP4DynamicActivationNVFP4WeightConfig
    m = nn.Sequential(nn.Linear(1024, 1024, bias=False)).cuda().bfloat16(); x = torch.randn(512, 1024, device="cuda", dtype=torch.bfloat16)
    ref = m(x); quantize_(m, NVFP4DynamicActivationNVFP4WeightConfig()); y = m(x)
    r = rel(y.float(), ref.float()); return r < 0.3, f"quantized-vs-bf16 output relative difference {r:.3f}"
try:
    import torchao  # noqa
    have_ao = True
except Exception:
    have_ao = False
if have_ao: gpu_check("torchao NVFP4 (dynamic activations + weights) inference is close to bf16", t_ao)
else: CHECKS.append(("torchao NVFP4", "SKIP", "torchao not installed")); print("SKIP - torchao not installed")

## Step 9 - Train a small network with the native GEMMs *(Blackwell only)*

A two-layer MLP student learns a random teacher (regression). Three runs from the same initialization and data: **fp32**, **recipe, emulated**, **recipe, native `scaled_mm`**. The native run should track the emulated one; both should track fp32 within the 4-bit noise. Sizes are multiples of 128 so every GEMM dimension is safe for the tensor cores.

In [ ]:
class PackedMLP(nn.Module):
    def __init__(self, backend, d=256, h=512):
        super().__init__(); self.backend = backend
        self.w1 = nn.Parameter(torch.randn(h, d) / math.sqrt(d)); self.w2 = nn.Parameter(torch.randn(d, h) / math.sqrt(h))
    def forward(self, x):
        cfg = k.RecipeConfig()
        if self.backend == "fp32": return F.gelu(x @ self.w1.t()) @ self.w2.t()
        return k.fp4_linear_packed(F.gelu(k.fp4_linear_packed(x, self.w1, cfg, self.backend)), self.w2, cfg, self.backend)

def t_train(steps=200):
    dev = "cuda"; res = {}
    for backend in ("fp32", "emulated", "scaled_mm"):
        torch.manual_seed(0); teacher = PackedMLP("fp32").to(dev).requires_grad_(False)
        torch.manual_seed(1); m = PackedMLP(backend).to(dev); opt = torch.optim.AdamW(m.parameters(), lr=2e-3)
        gen = torch.Generator(device=dev).manual_seed(123); torch.manual_seed(2); losses = []
        for _ in range(steps):
            xb = torch.randn(512, 256, device=dev, generator=gen)
            loss = F.mse_loss(m(xb), teacher(xb)); opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        res[backend] = losses
    f = {b: float(np.mean(l[-20:])) for b, l in res.items()}; s0 = res["fp32"][0]
    print({b: round(v, 5) for b, v in f.items()}, "| initial loss", round(s0, 4))
    ok = all(f[b] < 0.5 * s0 for b in f) and abs(f["scaled_mm"] - f["emulated"]) < 0.25 * s0
    return ok, f"final losses fp32 {f['fp32']:.4f} | emulated {f['emulated']:.4f} | native {f['scaled_mm']:.4f}"
gpu_check("MLP trains with native NVFP4/MXFP8 GEMMs and tracks the emulated recipe", t_train)

In [ ]:
print(f"{'#':>2s}  {'status':6s}  check")
for i, (n, s, d) in enumerate(CHECKS, 1): print(f"{i:2d}  {s:6s}  {n}" + (f"   [{d}]" if s != 'PASS' and d else ""))
n_pass = sum(c[1] == "PASS" for c in CHECKS); n_fail = sum(c[1] == "FAIL" for c in CHECKS); n_skip = sum(c[1] == "SKIP" for c in CHECKS)
print(f"\n{n_pass} passed, {n_fail} failed, {n_skip} skipped   | Blackwell GPU cells executed: {BLACKWELL}")
if not BLACKWELL: print("-> the GPU cells above were NOT run on this machine; treat them as unverified until run on Blackwell hardware.")
assert n_fail == 0, [c[0] for c in CHECKS if c[1] == "FAIL"]

## What to do with the results

- **All `PASS` on a Blackwell GPU:** the packed operands, the swizzled scales, the `scaled_mm` calls and the recipe layer are right; Step 7's table is your hardware's real FP4 / FP8 / bf16 throughput.
- **A `FAIL` in Step 5:** the printed relative difference tells you the kind of error. ≈ 1 means a layout problem (run the diagnostic cell: swapped nibbles / un-swizzled scales); a few percent means a precision/accumulation difference worth looking at. Exceptions are printed in full (typically unsupported dimension, dtype or architecture for that `scaled_mm` path).
- **Consumer GPUs (compute capability 12.x):** the FP4 tensor-core instructions exist, but which `scaled_mm` paths PyTorch exposes there depends on the version; the guarded cells report what works.

## What this notebook does not cover

The post's other hardware-level parts - the fused MoE kernels, PTX `cvt` conversions for S2F6/RHT in the GEMM epilogue, and quantized dispatch/combine over NVLink - need custom kernels and multiple GPUs. Their *numerics* (S2F6, RHT, stochastic rounding, 4/6) are tested in notebook 01. Also not covered: the speed-up and compute-efficiency figures the post reports for its large MoE run (≈ 8 % pretraining, ≈ 20 % RL wall-clock; ≈ 2 % compute-efficiency cost); a microbenchmark of one GEMM says little about an end-to-end training run.

### Honest limits
Never executed on Blackwell by the author of this notebook (Mac only); API names for `scaled_mm`, Transformer Engine and torchao are taken from PyTorch's installed source and the libraries' documentation; the meta-device check is shape-only; swizzle and packing were verified against PyTorch's reference helpers on the CPU.